In [3]:
"""
Analysis of the Hockey Environment.

Author: Jannik Rombach, Adriano Polzer

Reference:
    - Hockey Environment Source Code, 
    (hockey_env.py): https://github.com/martius-lab/hockey-env/blob/master/hockey/hockey_env.py

"""

import gymnasium as gym
import hockey.hockey_env as hockey_env
import numpy as np
import time


In [104]:
"""
1. Observation Space Structure

Reference:
    - __init__(); _get_obs(); obs_agent_two()
"""

# Create environment
env = gym.make('Hockey-v0')

print("="*70)
print("OBSERVATION SPACE ANALYSIS")
print("="*70)

# Query basic properties
print(f"Shape: {env.observation_space.shape}")
print(f"Dtype: {env.observation_space.dtype}")

# Get sample observation
obs, info = env.reset()
print(f"\nSample Observation Shape: {obs.shape}")
print(f"Sample values: {obs}")

# Document structure
print("\n" + "-"*70)
print("Observation Structure (from hockey_env.py lines 74-91):")
print("-"*70)

obs_structure = [
    (0, 1, "Player 1 position (x, y)", "relative to center"),
    (2, 2, "Player 1 angle", "rotation"),
    (3, 4, "Player 1 velocity (vx, vy)", "linear velocity"),
    (5, 5, "Player 1 angular velocity", "rotation speed"),
    (6, 7, "Player 2 position (x, y)", "relative to center"),
    (8, 8, "Player 2 angle", "rotation"),
    (9, 10, "Player 2 velocity (vx, vy)", "linear velocity"),
    (11, 11, "Player 2 angular velocity", "rotation speed"),
    (12, 13, "Puck position (x, y)", "relative to center"),
    (14, 15, "Puck velocity (vx, vy)", "linear velocity"),
    (16, 16, "Player 1 puck possession timer", "0-15 steps"),
    (17, 17, "Player 2 puck possession timer", "0-15 steps"),
]

print(f"{'Indices':<10} {'Component':<35} {'Notes':<25}")
print("-"*70)
for start, end, component, notes in obs_structure:
    if start == end:
        indices = f"[{start}]"
    else:
        indices = f"[{start}:{end+1}]"
    print(f"{indices:<10} {component:<35} {notes:<25}")

env.close()

OBSERVATION SPACE ANALYSIS
Shape: (18,)
Dtype: float32

Sample Observation Shape: (18,)
Sample values: [-3.          0.          0.          0.          0.          0.
  3.          0.          0.          0.          0.          0.
  1.62549782  0.92632341  0.          0.          0.          0.        ]

----------------------------------------------------------------------
Observation Structure (from hockey_env.py lines 74-91):
----------------------------------------------------------------------
Indices    Component                           Notes                    
----------------------------------------------------------------------
[0:2]      Player 1 position (x, y)            relative to center       
[2]        Player 1 angle                      rotation                 
[3:5]      Player 1 velocity (vx, vy)          linear velocity          
[5]        Player 1 angular velocity           rotation speed           
[6:8]      Player 2 position (x, y)            relative to

In [105]:
"""
2. Action Space Structure

Reference:
    - __init__(); step(); _apply_translate_action_with_max_speed(); _apply_rotation_action_with_max_speed(); _shoot()

"""

env = gym.make('Hockey-v0')

print("="*70)
print("ACTION SPACE ANALYSIS")
print("="*70)

# Query action space
print(f"Shape: {env.action_space.shape}")
print(f"Dtype: {env.action_space.dtype}")
print(f"Low:  {env.action_space.low}")
print(f"High: {env.action_space.high}")

# Sample action
sample_action = env.action_space.sample()
print(f"\nSample random action: {sample_action}")

# Document structure
print("\n" + "-"*70)
print("Action Structure:")
print("-"*70)

action_structure = [
    (0, "Player 1 force x", "[-1, 1]", "Horizontal force (left/right)"),
    (1, "Player 1 force y", "[-1, 1]", "Vertical force (up/down)"),
    (2, "Player 1 torque", "[-1, 1]", "Rotation torque"),
    (3, "Player 1 shoot", "[-1, 1]", "Shoot action (>0.5 triggers)"),
    (4, "Player 2 force x", "[-1, 1]", "Horizontal force"),
    (5, "Player 2 force y", "[-1, 1]", "Vertical force"),
    (6, "Player 2 torque", "[-1, 1]", "Rotation torque"),
    (7, "Player 2 shoot", "[-1, 1]", "Shoot action"),
]

print(f"{'Index':<8} {'Component':<20} {'Range':<12} {'Description':<30}")
print("-"*70)
for idx, component, range_val, desc in action_structure:
    print(f"{idx:<8} {component:<20} {range_val:<12} {desc:<30}")

print("As Player 1, control indices [0:4] only.")

env.close()

ACTION SPACE ANALYSIS
Shape: (8,)
Dtype: float32
Low:  [-1. -1. -1. -1. -1. -1. -1. -1.]
High: [1. 1. 1. 1. 1. 1. 1. 1.]

Sample random action: [-0.11958063 -0.15915495  0.4550779   0.90640664 -0.41391808 -0.64447796
 -0.15211824 -0.3468706 ]

----------------------------------------------------------------------
Action Structure:
----------------------------------------------------------------------
Index    Component            Range        Description                   
----------------------------------------------------------------------
0        Player 1 force x     [-1, 1]      Horizontal force (left/right) 
1        Player 1 force y     [-1, 1]      Vertical force (up/down)      
2        Player 1 torque      [-1, 1]      Rotation torque               
3        Player 1 shoot       [-1, 1]      Shoot action (>0.5 triggers)  
4        Player 2 force x     [-1, 1]      Horizontal force              
5        Player 2 force y     [-1, 1]      Vertical force                
6      

In [106]:
"""
3. Reward Structure Analysis

Reference:
    - _compute_reward(); get_reward(); _get_info(); _get_info_agent_two(); ContactDetector.BeginContact()

"""

print("="*70)
print("REWARD STRUCTURE ANALYSIS")
print("="*70)

# Document reward structure from source code
print("\n" + "-"*70)
print("Primary Rewards:")
print("-"*70)
print("Event               | Reward | Trigger")
print("-"*70)
print("Goal scored (win)   | +10    | self.winner == 1")
print("Goal conceded (loss)| -10    | self.winner == -1")
print("Episode timeout     |   0    | self.winner == 0")

print("\n" + "-"*70)
print("Default Reward Composition:")
print("-"*70)
print("reward = _compute_reward() + info['reward_closeness_to_puck']")

print("\n" + "-"*70)
print("Auxiliary Rewards in info dict:")
print("-"*70)
print("Key                          | Description")
print("-"*70)
print("reward_closeness_to_puck     | Penalty for distance to puck when defending")
print("reward_touch_puck            | +1 when touching puck, 0 otherwise")
print("reward_puck_direction        | Based on puck velocity toward goal")
print("winner                       | Game outcome: 1=win, -1=loss, 0=ongoing")

# Collect empirical data
print("\n" + "-"*70)
print("Empirical Analysis (5 episodes with random actions):")
print("-"*70)

env = gym.make('Hockey-v0')

reward_components = {
    'total_reward': [],
    'reward_closeness_to_puck': [],
    'reward_touch_puck': [],
    'reward_puck_direction': []
}

outcomes = []

for ep in range(5):
    obs, info = env.reset()
    done = False
    step = 0
    episode_reward = 0
    
    while not done and step < 250:
        action = env.action_space.sample()
        obs, reward, done, truncated, info = env.step(action)
        
        episode_reward += reward
        reward_components['total_reward'].append(reward)
        reward_components['reward_closeness_to_puck'].append(info['reward_closeness_to_puck'])
        reward_components['reward_touch_puck'].append(info['reward_touch_puck'])
        reward_components['reward_puck_direction'].append(info['reward_puck_direction'])
        
        step += 1
        done = done or truncated
    
    outcomes.append((ep+1, step, episode_reward, info['winner']))

env.close()

# Display
print("\nEpisode Results:")
print(f"{'Ep':<4} {'Steps':<8} {'Total Reward':<15} {'Outcome':<10}")
print("-"*40)
for ep, steps, reward, winner in outcomes:
    outcome = "Win" if winner == 1 else ("Loss" if winner == -1 else "Tie")
    print(f"{ep:<4} {steps:<8} {reward:<15.2f} {outcome:<10}")

REWARD STRUCTURE ANALYSIS

----------------------------------------------------------------------
Primary Rewards:
----------------------------------------------------------------------
Event               | Reward | Trigger
----------------------------------------------------------------------
Goal scored (win)   | +10    | self.winner == 1
Goal conceded (loss)| -10    | self.winner == -1
Episode timeout     |   0    | self.winner == 0

----------------------------------------------------------------------
Default Reward Composition:
----------------------------------------------------------------------
reward = _compute_reward() + info['reward_closeness_to_puck']

----------------------------------------------------------------------
Auxiliary Rewards in info dict:
----------------------------------------------------------------------
Key                          | Description
----------------------------------------------------------------------
reward_closeness_to_puck     | Penalt

In [2]:
"""
2. Observation Space Analysis (State Vector)

Reference:
    - _get_obs(); self.player1; self.player2; self.puck; self.keep_mode

The observation is a flattened numpy array (16 or 18 dimensions).
All positions are centered by subtracting [CENTER_X, CENTER_Y].
"""

print("="*70)
print("OBSERVATION STRUCTURE ANALYSIS")
print("="*70)

print("\n" + "-"*70)
print("Standard Observation Vector (16 Dimensions):")
print("-"*70)
print("Index | Description              | Source Attribute")
print("-"*70)
print("0, 1  | Player 1 Pos (x, y)      | self.player1.position")
print("2     | Player 1 Angle           | self.player1.angle")
print("3, 4  | Player 1 LinVel (vx, vy) | self.player1.linearVelocity")
print("5     | Player 1 AngVel          | self.player1.angularVelocity")
print("6, 7  | Player 2 Pos (x, y)      | self.player2.position")
print("8     | Player 2 Angle           | self.player2.angle")
print("9, 10 | Player 2 LinVel (vx, vy) | self.player2.linearVelocity")
print("11    | Player 2 AngVel          | self.player2.angularVelocity")
print("12, 13| Puck Position (x, y)     | self.puck.position")
print("14, 15| Puck LinVel (vx, vy)     | self.puck.linearVelocity")

print("\n" + "-"*70)
print("Conditional Observations (Keep Mode only):")
print("-"*70)
print("Index | Description              | Condition")
print("-"*70)
print("16    | Player 1 Has Puck Timer  | if self.keep_mode == True")
print("17    | Player 2 Has Puck Timer  | if self.keep_mode == True")

print("\n" + "-"*70)
print("Coordinate System Note:")
print("-"*70)
print("Field Center: [0, 0]")
print("Own Goal:     Negative X direction (approx. -4.0)")
print("Opponent Goal: Positive X direction (approx. +4.0)")

OBSERVATION STRUCTURE ANALYSIS

----------------------------------------------------------------------
Standard Observation Vector (16 Dimensions):
----------------------------------------------------------------------
Index | Description              | Source Attribute
----------------------------------------------------------------------
0, 1  | Player 1 Pos (x, y)      | self.player1.position
2     | Player 1 Angle           | self.player1.angle
3, 4  | Player 1 LinVel (vx, vy) | self.player1.linearVelocity
5     | Player 1 AngVel          | self.player1.angularVelocity
6, 7  | Player 2 Pos (x, y)      | self.player2.position
8     | Player 2 Angle           | self.player2.angle
9, 10 | Player 2 LinVel (vx, vy) | self.player2.linearVelocity
11    | Player 2 AngVel          | self.player2.angularVelocity
12, 13| Puck Position (x, y)     | self.puck.position
14, 15| Puck LinVel (vx, vy)     | self.puck.linearVelocity

-------------------------------------------------------------------

## Hockey render game

In [35]:
from time import sleep

import numpy as np
import hockey.hockey_env as h_env
import sys
import yaml
import os

sys.path.append(os.path.abspath(".."))
from agents.sac_agent import SACAgent

def test_agent(model_path, config_path_1, opponent_type="basic", opponent_model_path=None, config_path_2=None, num_episodes=5):
    with open(config_path_1, 'r') as f:
        config_1 = yaml.safe_load(f)

    env = h_env.HockeyEnv(mode=h_env.Mode.NORMAL)
    
    state_dim = env.observation_space.shape[0]
    action_dim = 4
    max_action = float(env.action_space.high[0])

    # Player 1
    agent = SACAgent(state_dim, action_dim, max_action, config_1)
    agent.load(model_path)

    # Player 2
    if opponent_type == "agent":
        with open(config_path_2, 'r') as f:
            config_2 = yaml.safe_load(f)
        player2 = SACAgent(state_dim, action_dim, max_action, config_2)
        if opponent_model_path:
            player2.load(opponent_model_path)
            print(f"Opponent: Agent {opponent_model_path}")
    else:
        player2 = h_env.BasicOpponent(weak=True)
        print("Opponent: BasicOpponent")
    for episode in range(num_episodes):
        obs, info = env.reset()
        done = False
        total_reward = 0
        
        while not done:
            env.render()
            
            a1 = agent.select_action(obs, eval_mode=True)
            
            obs_agent2 = env.obs_agent_two()
            if opponent_type == "agent":
                a2 = player2.select_action(obs_agent2, eval_mode=True)
            else:
                a2 = player2.act(obs_agent2) 
            
            obs, reward, terminated, truncated, info = env.step(np.hstack([a1, a2]))
            total_reward += reward
            done = terminated or truncated
            
        print(f"Episode {episode+1}: Reward = {total_reward:.2f} | Winner: {info.get('winner', 0)}")
    
    env.close()


In [41]:
test_agent("../logs/selfplay_v1/logs/checkpoint4_selfplay_baseline/agent_step_4000000.pth", "../logs/selfplay_v1/sac_job1.yaml", "agent", "../logs/baseline/checkpoint3_strong_er_baseline/agent_final.pth", "../configs/sac/cluster/checkpoint3_strong_baseline_er.yaml", num_episodes=15)

Using device: mps
Using device: mps
Opponent: Agent ../logs/baseline/checkpoint3_strong_er_baseline/agent_final.pth
Episode 1: Reward = 7.60 | Winner: 1
Episode 2: Reward = 9.15 | Winner: 1
Episode 3: Reward = 7.76 | Winner: 1
Episode 4: Reward = 8.21 | Winner: 1
Episode 5: Reward = -10.75 | Winner: -1
Episode 6: Reward = 9.24 | Winner: 1
Episode 7: Reward = 9.97 | Winner: 1
Episode 8: Reward = 9.00 | Winner: 1
Episode 9: Reward = -15.39 | Winner: -1
Episode 10: Reward = 8.50 | Winner: 1
Episode 11: Reward = 8.70 | Winner: 1
Episode 12: Reward = 8.78 | Winner: 1
Episode 13: Reward = -12.93 | Winner: -1
Episode 14: Reward = 9.58 | Winner: 1
Episode 15: Reward = 9.92 | Winner: 1


## Find Goal Center

In [24]:

SCALE = 60.0
VIEWPORT_W = 600
VIEWPORT_H = 480
H = VIEWPORT_H / SCALE
W = VIEWPORT_W / SCALE
CENTER_X = W / 2  

VIEWPORT_GOAL_X = W / 2 - 245 / SCALE - 10 / SCALE
OBS_GOAL_X = VIEWPORT_GOAL_X - CENTER_X
env = hockey_env.HockeyEnv()
for episode in range(1): 
    obs, info = env.reset()
    done = False

    env.render()
    player_pos = obs[0:2] 

    goal_center = np.array([OBS_GOAL_X, 0.0]) 
    dist_to_goal = np.linalg.norm(player_pos - goal_center)
    print("Player position: ", player_pos)
    print("Goal env: ", OBS_GOAL_X)
    print("Center X: ", CENTER_X)
    print(f"Distance to goal: {dist_to_goal:.2f}")


Player position:  [-3.  0.]
Goal env:  -4.25
Center X:  5.0
Distance to goal: 1.25
